# 1. Base Model Training

Train an **XGBoost** model on the SMOTENC-resampled training data (Experiment 3 strategy) and evaluate it on the held-out test split using the optimal decision threshold of **0.70**.

Results from Experiment 3 (SMOTENC + Threshold Tuning @ 0.70):
- **Precision**: 0.82
- **Recall**: 0.80
- **F1-Score**: 0.81
- **ROC AUC**: 0.9980

## 1. Imports & Setup

In [1]:
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import xgboost as xgb
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings('ignore')

# Keep the run reproducible.
SEED = 42
np.random.seed(SEED)

print('Imports and seed set.')
print(f'XGBoost version: {xgb.__version__}')

Imports and seed set.
XGBoost version: 3.2.0


## 2. Load Prepared Artifacts

Read the SMOTENC-resampled train/test arrays and the optimal threshold produced by `0_data_preparation.ipynb`.

In [2]:
# Resolve repository root
BASE_DIR = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / 'artifacts').exists() or (candidate / 'requirements.txt').exists() or (candidate / 'README.md').exists():
        BASE_DIR = candidate
        break
if BASE_DIR is None:
    BASE_DIR = Path.cwd()
    print('Warning: repository root not found; using CWD as BASE_DIR')

ARTIFACT_DIR = BASE_DIR / 'artifacts' / 'data'
EVAL_DIR     = BASE_DIR / 'artifacts' / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(
        f'Artifact directory not found: {ARTIFACT_DIR}. Run 0_data_preparation.ipynb first.'
    )

# Load split arrays
X_train = np.load(ARTIFACT_DIR / 'credit_card_fraud_X_train.npz')['X_train']
X_test  = np.load(ARTIFACT_DIR / 'credit_card_fraud_X_test.npz')['X_test']
y_train = np.load(ARTIFACT_DIR / 'credit_card_fraud_y_train.npz')['y_train']
y_test  = np.load(ARTIFACT_DIR / 'credit_card_fraud_y_test.npz')['y_test']

# Load feature names and optimal threshold
with open(ARTIFACT_DIR / 'features.json', 'r') as f:
    feature_names = json.load(f)

with open(ARTIFACT_DIR / 'threshold.json', 'r') as f:
    OPTIMAL_THRESHOLD = json.load(f)['optimal_threshold']

# Wrap as DataFrames for readability
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df  = pd.DataFrame(X_test,  columns=feature_names)

print(f'BASE_DIR        : {BASE_DIR}')
print(f'EVAL_DIR        : {EVAL_DIR}')
print(f'X_train shape   : {X_train.shape}')
print(f'X_test  shape   : {X_test.shape}')
print(f'y_train shape   : {y_train.shape}  (fraud={int(y_train.sum()):,})')
print(f'y_test  shape   : {y_test.shape}  (fraud={int(y_test.sum()):,})')
print(f'Optimal threshold: {OPTIMAL_THRESHOLD}')

BASE_DIR        : C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system
EVAL_DIR        : C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\evaluation
X_train shape   : (1134468, 29)
X_test  shape   : (259335, 29)
y_train shape   : (1134468,)  (fraud=103,133)
y_test  shape   : (259335,)  (fraud=1,501)
Optimal threshold: 0.7


## 3. Train XGBoost Model

Fit an XGBoost classifier on the SMOTENC-resampled training split.

In [3]:
model = xgb.XGBClassifier(
    random_state=SEED,
    eval_metric='logloss',
    n_estimators=100,
    max_depth=6,
    n_jobs=-1,
)
model.fit(X_train_df, y_train)
print('XGBoost model trained successfully.')

XGBoost model trained successfully.


## 4. Make Predictions

Generate fraud probabilities, then apply the optimal threshold of **0.70** (Experiment 3) to produce the final class labels.

In [4]:
# Probabilities
y_proba_train = model.predict_proba(X_train_df)[:, 1]
y_proba_test  = model.predict_proba(X_test_df)[:, 1]

# Apply optimal threshold
y_hat_train = (y_proba_train >= OPTIMAL_THRESHOLD).astype(int)
y_hat_test  = (y_proba_test  >= OPTIMAL_THRESHOLD).astype(int)

print(f'Threshold applied : {OPTIMAL_THRESHOLD}')
print(f'Predicted fraud in test set: {int(y_hat_test.sum()):,}')

Threshold applied : 0.7
Predicted fraud in test set: 1,463


## 5. Evaluate Performance

Measure the model with standard classification metrics on the test split.

In [5]:
accuracy  = accuracy_score(y_test, y_hat_test)
precision = precision_score(y_test, y_hat_test)
recall    = recall_score(y_test, y_hat_test)
f1        = f1_score(y_test, y_hat_test)
roc_auc   = roc_auc_score(y_test, y_proba_test)

print('--- Test Set Metrics ---')
print(f'Accuracy  : {accuracy:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1-Score  : {f1:.4f}')
print(f'ROC AUC   : {roc_auc:.4f}')
print()
print('--- Full Classification Report ---')
print(classification_report(y_test, y_hat_test, target_names=['Legitimate', 'Fraud']))

--- Test Set Metrics ---
Accuracy  : 0.9979
Precision : 0.8243
Recall    : 0.8035
F1-Score  : 0.8138
ROC AUC   : 0.9980

--- Full Classification Report ---
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00    257834
       Fraud       0.82      0.80      0.81      1501

    accuracy                           1.00    259335
   macro avg       0.91      0.90      0.91    259335
weighted avg       1.00      1.00      1.00    259335



## 6. Confusion Matrix

Visualize the test-set classification outcomes.

In [6]:
cm = confusion_matrix(y_test, y_hat_test)
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Legitimate', 'Fraud'],
    yticklabels=['Legitimate', 'Fraud'],
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix (threshold={OPTIMAL_THRESHOLD})')
plt.tight_layout()
plt.savefig(EVAL_DIR / 'confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved.')

Confusion matrix saved.


## 7. Feature Importance

Identify the top features that drive the model's predictions.

In [7]:
importance = model.feature_importances_
feat_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
feat_df = feat_df.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=feat_df.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top-15 Feature Importances — XGBoost')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(EVAL_DIR / 'feature_importance.png', dpi=150)
plt.show()
print('Feature importance plot saved.')
print(feat_df.head(15).to_string(index=False))

Feature importance plot saved.
                        feature  importance
        transaction_hour_binned    0.205402
merchant_category_gas_transport    0.178464
                         amount    0.149551
  merchant_category_grocery_pos    0.088261
         merchant_category_home    0.046320
  merchant_category_grocery_net    0.037214
          velocity_last_24h_log    0.034145
  merchant_category_food_dining    0.033586
 merchant_category_shopping_net    0.029360
merchant_category_entertainment    0.029011
       merchant_category_travel    0.027128
                     amount_log    0.026021
 merchant_category_shopping_pos    0.023346
     merchant_category_misc_pos    0.022328
     merchant_category_misc_net    0.020619


## 8. Save the Trained Model

Persist the trained model to `artifacts/models/` for use in downstream notebooks and the training pipeline.

In [8]:
MODEL_DIR = BASE_DIR / 'artifacts' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / 'xgboost_fraud_model.pkl'
joblib.dump(model, model_path)

print(f'Model saved to: {model_path}')

Model saved to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\models\xgboost_fraud_model.pkl
